# 05 - Time-delay and identification-epoch distributions

Compares the two epochs the analysis produces for each lensed system:

* **`time_of_SN`** - the rest-frame epoch (relative to the intrinsic peak) at which
  the *colour method* could flag the system from a single magnified image;
* **`time_delay`** - the rest-frame epoch at which the *delayed second image* is
  first detected, i.e. when image multiplicity becomes apparent.

Both are measured against the same rest-frame intrinsic peak day, which is what
makes them directly comparable. If the colour method fires systematically earlier,
that is the case for using it as a trigger.

Run notebook 04 first to produce `cadence_population.npz`.

In [ ]:
# If running in Colab, install the dependencies (uncomment):
# !pip install sncosmo
# !pip install git+https://github.com/LSSTDESC/OpSimSummaryV2.git

# Make the `cmsne` package importable when this notebook lives in notebooks/.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from cmsne.colour_magnitude import (modified_weighted_vals, weighted_fraction,
                                    weighted_quantile)

In [ ]:
data = np.load('cadence_population.npz')
CM_times      = data['l_times']
time_delays   = data['time_delays']     # epoch the 2nd image is first detected
timdel        = data['timdel']          # physical lensing delay, for validation
redshifts     = data['redshifts']
magnifications = data['magnifications']
weights_l     = data['weights_l']

print(f"{len(time_delays)} events loaded")

# Keep every array index-aligned: build ONE mask over all of them rather than
# filtering each array separately, which would silently pair a time delay with a
# different event's weight.
finite = (np.isfinite(CM_times) & np.isfinite(time_delays) & np.isfinite(redshifts)
          & np.isfinite(magnifications) & np.isfinite(weights_l) & np.isfinite(timdel))
print(f"{finite.sum()} events with all quantities finite")

CM_times, time_delays = CM_times[finite], time_delays[finite]
redshifts, magnifications = redshifts[finite], magnifications[finite]
weights_l, timdel = weights_l[finite], timdel[finite]

## Validate before interpreting

The second-image epoch is only meaningful if it actually carries the physical delay.
An earlier version of the pipeline re-zeroed each light curve's time axis on its first
visit, which erased the delay and pinned this quantity near the model's minimum phase
-- it correlated with magnification at -0.09 and differed by only 2 days between
mu<10 and mu>40, despite the delay scaling as mu^-3. Check that before reading
anything off the histograms.

In [ ]:
r_delay = np.corrcoef(time_delays, timdel)[0, 1]
r_mu    = np.corrcoef(time_delays, magnifications)[0, 1]
print(f"corr(second-image epoch, physical delay) = {r_delay:+.4f}   (want >> 0)")
print(f"corr(second-image epoch, magnification)  = {r_mu:+.4f}   (want < 0, delay ~ mu^-3)")

lo, hi = magnifications < 10, magnifications > 40
print(f"median epoch, mu<10 : {np.median(time_delays[lo]):+7.2f} d  (n={lo.sum()})")
print(f"median epoch, mu>40 : {np.median(time_delays[hi]):+7.2f} d  (n={hi.sum()})")
print(f"physical delay range: {timdel.min():.3f} to {timdel.max():.1f} d")

if r_delay < 0.5:
    print("\nWARNING: the epoch does not track the physical delay - do not interpret "
          "the histograms below.")
else:
    print("\nOK: the epoch tracks the physical delay.")

## Rate-weighted resampling

In [ ]:
N_DRAW = 1000

(weight_time_delays, new_CM_times, new_magnifications,
 new_redshift, normalized_weights) = modified_weighted_vals(
    weights_l, time_delays, CM_times, magnifications, redshifts, N_DRAW)

print(f"resampled to {len(weight_time_delays)} events")

## Identification epoch vs image-multiplicity epoch

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(-40, 150, 60)
ax.hist(new_CM_times, bins=bins, alpha=0.55, label='colour method (`time_of_SN`)')
ax.hist(weight_time_delays, bins=bins, alpha=0.55, label='second image (`time_delay`)')
ax.set_xlabel('Rest-frame epoch relative to intrinsic peak (days)')
ax.set_ylabel('Rate-weighted counts')
ax.legend()
plt.show()

# Headline numbers come from the exact weighted estimators on the FULL population,
# not from the resampled draw above. Resampling is only needed to draw the
# histogram; a mean, a fraction or a quantile can be weighted directly, which is
# exact and free of Monte-Carlo noise.
print(f"median colour-method epoch : {weighted_quantile(CM_times, weights_l):+.2f} d")
print(f"median second-image epoch  : {weighted_quantile(time_delays, weights_l):+.2f} d")
print(f"colour method earlier in "
      f"{100 * weighted_fraction(CM_times < time_delays, weights_l):.1f}% of systems")
print(f"  (unweighted, for comparison: "
      f"{100 * np.mean(CM_times < time_delays):.1f}%)")

## How many of these delays are actually measurable?

The magnification prior is flat in mu over 2-50, but the delay scales as
`(mu/4)**-3`, so most of that range maps to sub-day delays. For those systems the
"second image" is detected essentially simultaneously with the first, which is not
an image-multiplicity detection in any observational sense -- you would need to
resolve the pair spatially, and the angular-separation cut in `generate_one`
(`125 / mu` arcsec against a 0.8 arcsec threshold) never fires for mu <= 50.

Read the epoch comparison above against this table before drawing conclusions.

In [ ]:
print(f"{'delay cut':>12} {'share':>8} {'colour earlier':>16} {'median 2nd image':>18}")
for thr in [0.0, 1.0, 2.0, 3.0, 5.0, 10.0, 20.0]:
    m = timdel > thr
    if m.sum() < 50:
        continue
    frac = weighted_fraction(CM_times[m] < time_delays[m], weights_l[m])
    med = weighted_quantile(time_delays[m], weights_l[m])
    print(f"{thr:>10.0f} d {100 * m.mean():>7.1f}% {100 * frac:>15.1f}% {med:>16.2f} d")

print(f"\nmedian physical delay: {np.median(timdel):.2f} d; "
      f"{100 * np.mean(timdel < 1):.0f}% are under 1 day")

## Per-system offset between the two identification epochs

The two epochs above are marginal distributions. The quantity that actually matters
operationally is the **paired** difference for each system:

    lead = time_delay - time_of_SN

i.e. how many rest-frame days earlier the colour method flags a system than its second
image becomes detectable. Positive means the colour method gets there first.

Read the regime split alongside it. Where the delay exceeds the supernova's own visible
lifetime the colour method wins by construction -- the second image simply has not
arrived yet -- so the overall fraction is dominated by systems where multiplicity is not
yet available, rather than by the colour method beating a live alternative.

In [ ]:
import sncosmo

lead = time_delays - CM_times          # + => colour method identifies first

# Classify by how long the delay is relative to the SN's own visible lifetime,
# since that is what sets the sign of the offset.
src = sncosmo.get_source('salt3')
visible = (src.maxphase() - src.minphase()) * (1 + redshifts)
ratio = timdel / visible

regimes = [('delay > light curve',    ratio >= 1,                        '#2c7d5c'),
           ('10-100% of light curve', (ratio >= .1) & (ratio < 1),       '#8ab17d'),
           ('1 d - 10%',              (timdel >= 1) & (ratio < .1),      '#e0a851'),
           ('delay < 1 d',            timdel < 1,                        '#b1372b')]

LO, HI, NBINS = -60, 200, 66
edges = np.linspace(LO, HI, NBINS)
clipped = np.clip(lead, LO + 1e-9, HI - 1e-9)   # outermost bins are overflow

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5), gridspec_kw={'width_ratios': [2, 1]})

ax = axes[0]
counts, _, patches = ax.hist(
    [clipped[m] for _, m, _ in regimes], bins=edges, stacked=True,
    weights=[weights_l[m] / weights_l.sum() for _, m, _ in regimes],
    color=[c for _, _, c in regimes], label=[l for l, _, _ in regimes],
    edgecolor='white', linewidth=.2)

# The end bins pile up everything beyond the axis; hatch them so the spike is not
# mistaken for real structure at +200 d.
for group in patches:
    for bar in (group[0], group[-1]):
        bar.set_hatch('///')
        bar.set_edgecolor('0.35')

ax.axvline(0, color='k', lw=1.4)
med = weighted_quantile(lead, weights_l)
ax.axvline(med, color='#b5179e', lw=1.6, ls='--', label=f'weighted median {med:+.0f} d')
frac = 100 * weighted_fraction(lead > 0, weights_l)
ax.annotate(f'colour method first\n{frac:.1f}%', xy=(.60, .93), xycoords='axes fraction',
            ha='center', fontsize=11, color='#2c7d5c', fontweight='bold')
ax.annotate(f'multiplicity first\n{100 - frac:.1f}%', xy=(.16, .93), xycoords='axes fraction',
            ha='center', fontsize=11, color='#b1372b', fontweight='bold')
over = 100 * weighted_fraction(lead >= HI, weights_l)
under = 100 * weighted_fraction(lead <= LO, weights_l)
ax.set_xlim(LO, HI)
ax.set_xlabel('Colour method lead time (rest-frame days)\n'
              'time_delay - time_of_SN;  positive = colour method identifies first')
ax.set_ylabel('Rate-weighted fraction per bin')
ax.set_title('Offset between the two identification epochs\n'
             f'hatched end bins are overflow: {under:.1f}% below, {over:.1f}% above',
             fontsize=11)
ax.legend(fontsize=8.5, loc='center right')

ax = axes[1]
labels = [l for l, _, _ in regimes]
shares = [100 * weighted_fraction(m, weights_l) for _, m, _ in regimes]
meds = [weighted_quantile(lead[m], weights_l[m]) for _, m, _ in regimes]
ypos = np.arange(len(labels))
ax.barh(ypos, meds, color=[c for _, _, c in regimes], height=.62)
ax.axvline(0, color='k', lw=1.2)
ax.set_yticks(ypos)
ax.set_yticklabels([f'{l}\n({s:.0f}% of systems)' for l, s in zip(labels, shares)], fontsize=8.5)
ax.invert_yaxis()
for y, m_ in zip(ypos, meds):
    ax.annotate(f'{m_:+.0f} d', xy=(m_, y), xytext=(6 if m_ >= 0 else -6, 0),
                textcoords='offset points', va='center',
                ha='left' if m_ >= 0 else 'right', fontsize=9)
ax.set_xlabel('Median lead time (rest-frame days)')
ax.set_title('...split by how long the delay is', fontsize=11)
span = max(abs(min(meds)), abs(max(meds)))
ax.set_xlim(-0.35 * span, 1.30 * span)   # room for the value labels at both ends

plt.tight_layout()
plt.show()

print(f"weighted median lead      : {med:+.2f} d")
print(f"weighted IQR              : "
      f"{weighted_quantile(lead, weights_l, .25):+.1f} to "
      f"{weighted_quantile(lead, weights_l, .75):+.1f} d")
print(f"colour method identifies first in {frac:.1f}% of systems\n")
print(f"{'regime':<24}{'share':>8}{'median lead':>14}{'colour first':>14}")
for (lbl, m, _), share in zip(regimes, shares):
    print(f"{lbl:<24}{share:>7.1f}%{weighted_quantile(lead[m], weights_l[m]):>+13.1f} d"
          f"{100 * weighted_fraction(lead[m] > 0, weights_l[m]):>13.1f}%")

## Dependence on redshift and magnification

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

axes[0, 0].hist2d(new_CM_times, new_redshift, bins=50)
axes[0, 0].set_xlabel('Colour-method epoch (days)'); axes[0, 0].set_ylabel('Redshift')

axes[0, 1].hist2d(weight_time_delays, new_redshift, bins=50)
axes[0, 1].set_xlabel('Second-image epoch (days)'); axes[0, 1].set_ylabel('Redshift')

axes[1, 0].hist2d(new_magnifications, new_redshift, bins=50)
axes[1, 0].set_xlabel('Magnification'); axes[1, 0].set_ylabel('Redshift')

axes[1, 1].hist2d(new_CM_times, new_magnifications, bins=50)
axes[1, 1].set_xlabel('Colour-method epoch (days)'); axes[1, 1].set_ylabel('Magnification')

plt.tight_layout()
plt.show()